<a href="https://colab.research.google.com/github/beyzoskaya/proteinAllergen/blob/main/ESMFold_batches.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title Install ESMFold, OpenFold and download params
version = "1"  # use latest
model_name = "esmfold.model"
import os, time

if not os.path.isfile(model_name):
  # download esmfold params
  os.system("apt-get install aria2 -qq")
  os.system(f"aria2c -q -x 16 https://colabfold.steineggerlab.workers.dev/esm/{model_name} &")

  if not os.path.isfile("finished_install"):
    print("installing libs...")
    os.system("pip install -q omegaconf pytorch_lightning biopython ml_collections einops py3Dmol")
    os.system("pip install -q git+https://github.com/NVIDIA/dllogger.git")
    print("installing openfold...")
    os.system("pip install -q git+https://github.com/sokrypton/openfold.git")
    print("installing esmfold...")
    os.system("pip install -q git+https://github.com/sokrypton/esm.git")
    os.system("touch finished_install")

  # wait for params to finish downloading
  while not os.path.isfile(model_name):
    time.sleep(5)
  if os.path.isfile(f"{model_name}.aria2"):
    print("downloading params...")
  while os.path.isfile(f"{model_name}.aria2"):
    time.sleep(5)

print(" Installation done!")

installing libs...
installing openfold...
installing esmfold...
 Installation done!


In [3]:
!pip install -q omegaconf pytorch_lightning biopython ml_collections einops py3Dmol modelcif
!pip install -q git+https://github.com/NVIDIA/dllogger.git
!pip install -q git+https://github.com/sokrypton/openfold.git
!pip install -q git+https://github.com/sokrypton/esm.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 392.5/392.5 kB 10.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import torch
import gc
import re
from jax.tree_util import tree_map
from scipy.special import softmax
import numpy as np

# load model
model = torch.load(model_name, weights_only=False)
model.eval().cuda().requires_grad_(False)
model_name_ = model_name

# set chunk size based on average protein length
model.set_chunk_size(128)

torch.cuda.empty_cache()
print(" Model loaded")

 Model loaded


In [8]:
!fusermount -u /content/drive

# Remove anything left inside the mount point
!rm -rf /content/drive

fusermount: failed to unmount /content/drive: Invalid argument


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
from Bio import SeqIO
import os

fasta_path = "/content/drive/MyDrive/proteinAllergenDataset/all.test.fasta"

output_dir = "/content/drive/MyDrive/proteinAllergenDataset/esmfold_test_pdbs"
os.makedirs(output_dir, exist_ok=True)

records = list(SeqIO.parse(fasta_path, "fasta"))
print(f" Loaded {len(records)} sequences")

records = records[:5]
print(f" Will predict only first {len(records)} sequences")

 Loaded 1420 sequences
 Will predict only first 5 sequences


In [11]:
def parse_output(output):
    pae = (output["aligned_confidence_probs"][0] * np.arange(64)).mean(-1) * 31
    plddt = output["plddt"][0,:,1]

    bins = np.append(0,np.linspace(2.3125,21.6875,63))
    sm_contacts = softmax(output["distogram_logits"],-1)[0]
    sm_contacts = sm_contacts[...,bins<8].sum(-1)
    xyz = output["positions"][-1,0,:,1]
    mask = output["atom37_atom_exists"][0,:,1] == 1
    o = {"pae":pae[mask,:][:,mask],
         "plddt":plddt[mask],
         "sm_contacts":sm_contacts[mask,:][:,mask],
         "xyz":xyz[mask]}
    return o

In [12]:
num_recycles = 3    # change if you like
chain_linker = 25   # same as in original

MAX_LEN = 1500      # skip too long sequences

for idx, record in enumerate(records):
    seq = str(record.seq).upper()
    seq = re.sub("[^A-Z]", "", seq)

    if len(seq) == 0 or len(seq) > MAX_LEN:
        print(f"[{idx+1}/{len(records)}] Skipping: {record.id} (empty or too long: len={len(seq)})")
        continue

    print(f"[{idx+1}/{len(records)}] Predicting: {record.id} (len={len(seq)})")

    # run prediction
    output = model.infer(seq,
                         num_recycles=num_recycles,
                         chain_linker="X"*chain_linker,
                         residue_index_offset=512)

    pdb_str = model.output_to_pdb(output)[0]

    # save PDB
    pdb_filename = os.path.join(output_dir, f"{record.id}.pdb")
    with open(pdb_filename, "w") as f:
        f.write(pdb_str)

    # optional: parse and save PAE etc.
    output_np = tree_map(lambda x: x.cpu().numpy(), output)
    O = parse_output(output_np)
    ptm = output_np["ptm"][0]
    plddt = output_np["plddt"][0,...,1].mean()
    np.savetxt(os.path.join(output_dir, f"{record.id}_pae.txt"), O["pae"], "%.3f")

    print(f"  -> Saved PDB. ptm: {ptm:.3f}, plddt: {plddt:.3f}")

    del output
    torch.cuda.empty_cache()

print(" All done! PDBs saved to:", output_dir)

[1/5] Predicting: allergen_2145 (len=264)
  -> Saved PDB. ptm: 0.624, plddt: 78.487
[2/5] Predicting: allergen_109 (len=161)
  -> Saved PDB. ptm: 0.817, plddt: 87.004
[3/5] Predicting: allergen_2419 (len=333)
  -> Saved PDB. ptm: 0.933, plddt: 91.518
[4/5] Predicting: allergen_12 (len=481)
  -> Saved PDB. ptm: 0.847, plddt: 83.475
[5/5] Predicting: allergen_117 (len=320)
  -> Saved PDB. ptm: 0.935, plddt: 94.693
 All done! PDBs saved to: /content/drive/MyDrive/proteinAllergenDataset/esmfold_test_pdbs


In [13]:
from Bio.PDB import PDBParser

parser = PDBParser()
structure = parser.get_structure("allergen_2145", "/content/drive/MyDrive/proteinAllergenDataset/esmfold_test_pdbs/allergen_2145.pdb")

print(structure)

<Structure id=allergen_2145>


/usr/local/lib/python3.11/dist-packages/Bio/PDB/PDBParser.py:384: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 2039
  warnings.warn(


In [14]:
!head /content/drive/MyDrive/proteinAllergenDataset/esmfold_test_pdbs/allergen_2145.pdb

PARENT N/A
ATOM      1  N   MET A   1      25.655 -17.271  -5.545  1.00 51.00           N  
ATOM      2  CA  MET A   1      24.327 -17.879  -5.579  1.00 56.66           C  
ATOM      3  C   MET A   1      23.323 -16.955  -6.259  1.00 53.38           C  
ATOM      4  CB  MET A   1      23.853 -18.217  -4.164  1.00 46.19           C  
ATOM      5  O   MET A   1      22.315 -17.415  -6.798  1.00 51.22           O  
ATOM      6  CG  MET A   1      24.233 -19.616  -3.708  1.00 44.42           C  
ATOM      7  SD  MET A   1      23.637 -19.988  -2.013  1.00 51.78           S  
ATOM      8  CE  MET A   1      24.039 -21.755  -1.913  1.00 39.17           C  
ATOM      9  N   VAL A   2      23.599 -15.635  -6.555  1.00 64.74           N  


In [15]:
import py3Dmol

def show_pdb(pdb_file):
    with open(pdb_file) as f:
        pdb_data = f.read()
    view = py3Dmol.view(width=400, height=400)
    view.addModel(pdb_data, 'pdb')
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    view.zoomTo()
    return view.show()

show_pdb("/content/drive/MyDrive/proteinAllergenDataset/esmfold_test_pdbs/allergen_2145.pdb")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
zip_path = "/content/drive/MyDrive/esmfold_pdbs.zip"
!zip -r {zip_path} {output_dir}
print(f" Zipped to: {zip_path}")